# 에이전트 시스템 평가(Evaluation)

이 노트북은 agent 시스템을 "그럴듯해 보이는 데모"가 아니라 "측정 가능한 시스템"으로 다루는 방법을 설명한다. 질문 세트를 불러오고, baseline과 agent workflow를 반복 실행한 뒤, 여러 지표를 통해 어떤 부분이 좋아졌고 어떤 대가를 치렀는지 본다.

## 학습 목표
- evaluation dataset이 왜 필요한지, 어떤 분포를 가져야 하는지 이해한다.
- `answer_correctness`, `retrieval_hit_rate`, `grounding_pass_rate`, `abstain_precision`, `latency`, `average_steps`가 각각 무엇을 뜻하는지 설명할 수 있다.
- baseline과 agent workflow를 숫자로 비교하고, 서로의 trade-off를 읽는 법을 익힌다.
- 레이더 차트(radar chart)와 질문 유형별 breakdown을 면접에서 어떻게 해석할지 정리할 수 있다.


## 개념 설명

평가 노트북도 가장 먼저 실행 환경을 확인한다. evaluation은 보통 시간이 조금 더 걸리고, 아티팩트 파일까지 생성하므로, 잘못된 환경에서 돌리면 결과가 섞이거나 이전 파일을 엉뚱하게 덮어쓸 수 있다.

- **목적**: 현재 노트북이 어떤 Python 실행 파일과 runtime 설정을 사용하는지 명확히 확인한다.
- **핵심 로직**: 프로젝트 루트를 `sys.path`에 넣고, `RuntimeConfig.auto_detect()`로 디바이스/백엔드 설정을 출력한다.
- **주요 파라미터/변수**:
  - `ROOT`: 프로젝트 기준 경로이다.
  - `sys.executable`: 현재 커널의 실제 Python 경로이다.
  - `RuntimeConfig.auto_detect()`: evaluation이 어떤 장치 설정에서 돌고 있는지 보여준다.

평가에서 재현성은 특히 중요하다. 같은 코드를 다른 환경에서 돌렸을 때 결과가 조금씩 달라질 수 있기 때문에, 항상 시작점부터 기록해두는 것이 좋다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## 왜 evaluation이 중요한가

에이전트 시스템은 데모 한두 개만 보면 성능이 좋아 보이기 쉽다. 하지만 실제로는 질문 유형이 다양하고, retrieval은 맞는데 synthesis가 틀릴 수도 있고, grounding은 잘되지만 abstain을 너무 자주 할 수도 있다. evaluation은 이런 균형을 수치로 드러내는 장치다.

이 셀은 평가 데이터셋을 먼저 불러와, 우리가 무엇을 시험하려는지 범위를 파악한다.

- **목적**: 평가 질문의 전체 크기와 스키마를 확인한다.
- **핵심 로직**: `load_eval_dataset()`으로 JSON 데이터를 읽고, `dataset_frame`으로 정리해 질문 ID, 질문 유형, 기대 상태, 질문 본문을 확인한다.
- **주요 파라미터/변수**:
  - `dataset`: 로드된 평가 샘플 리스트이다.
  - `dataset_frame`: 평가셋을 표로 본 데이터프레임이다.
  - `expected_status`: 질문이 answered 되어야 하는지, 아니면 abstained 되어야 하는지 나타낸다.

출력에서 먼저 볼 것은 총 질문 수와 질문 유형 분포의 균형이다. 한 유형만 많으면 평균 점수가 높아도 시스템 전체 품질을 과대평가할 수 있다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluator import load_eval_dataset, run_evaluation_suite

dataset = load_eval_dataset()
dataset_frame = pd.DataFrame(dataset)
print(f'Total questions: {len(dataset_frame)}')
dataset_frame[['id', 'question_type', 'expected_status', 'question']].head(12)

## 평가 데이터셋 설계(Designing evaluation datasets)

좋은 평가셋은 단순히 질문 수가 많은 것이 아니라, 시스템이 실제로 부딪히는 문제 유형을 골고루 담아야 한다. 이 프로젝트에서는 `simple_lookup`, `comparison`, `multi_hop`, `summary`, `insufficient_evidence_risk`를 균형 있게 포함한다. 특히 abstain이 필요한 질문이 충분히 들어 있어야 verifier와 fallback의 품질을 시험할 수 있다.

평가셋을 설계할 때는 다음을 함께 본다.
- 쉬운 질문과 어려운 질문이 섞여 있는가
- 여러 문서를 조합해야 하는 multi-hop 질문이 있는가
- 문서 범위 밖 질문이 포함되어 있는가
- 특정 문서 하나에만 편향되어 있지 않은가

💡 면접 포인트: "평가셋이 곧 제품 요구사항의 축약판"이라고 설명하면 좋다. 시스템이 무엇을 잘해야 하는지 먼저 데이터셋으로 정의하는 셈이다.


## 구현

이 셀은 질문 유형 분포를 막대그래프로 그려서, 평가셋이 균형 있게 구성되어 있는지 확인한다. 숫자만 나열하는 것보다 시각화하면 어느 유형이 과하거나 부족한지 훨씬 빨리 읽을 수 있다.

- **목적**: 평가셋이 query type 기준으로 얼마나 균형 잡혀 있는지 시각적으로 확인한다.
- **핵심 로직**: `value_counts().sort_index()`로 유형별 개수를 세고, `plot(kind='bar')`로 막대그래프를 그린다.
- **주요 파라미터/변수**:
  - `distribution`: 질문 유형별 개수 시리즈이다.
  - `ax`: matplotlib 축 객체로, 축 라벨과 제목을 붙이는 데 쓴다.

이 차트를 읽을 때는 막대 높이가 비슷한지 보는 것이 핵심이다. 특정 유형만 과도하게 많으면 평균 metric이 그 유형에 끌려가므로, 모델 개선 방향을 잘못 해석할 수 있다.


In [ ]:
distribution = dataset_frame['question_type'].value_counts().sort_index()
ax = distribution.plot(kind='bar', color='#4C78A8', title='Evaluation Dataset Distribution by Query Type')
ax.set_xlabel('query_type')
ax.set_ylabel('question count')
plt.tight_layout()
plt.show()
distribution.reset_index().rename(columns={'index': 'question_type', 'question_type': 'count'})

## 지표(metrics)

이제 baseline과 agent workflow를 실제로 반복 실행하고, 평가 지표를 계산한다. 각 metric은 다른 실패 모드를 본다.

- `answer_correctness`: token F1 기반으로 정답과 예측 답변이 얼마나 겹치는지 본다. 표현이 조금 달라도 핵심 토큰이 맞으면 점수가 올라간다.
- `retrieval_hit_rate`: gold source가 top-k retrieval 안에 들어왔는지를 0 또는 1로 측정한다. retrieval 자체가 맞았는지 보는 지표다.
- `grounding_pass_rate`: verifier가 "근거 충분"으로 판정한 비율이다. 답변이 문서에 실제로 기대고 있는지 본다.
- `abstain_precision`: abstain한 케이스 중 실제로 abstain해야 했던 케이스의 비율이다. 겁이 많은 시스템인지, 보수적으로 정확한지 구분할 때 필요하다.
- `latency`: 한 질문을 처리하는 데 걸린 평균 시간이다.
- `average_steps`: workflow가 평균적으로 몇 단계의 trace를 남기는지 보여준다. 제어 복잡도의 대략적인 지표다.

- **목적**: 두 시스템을 동일한 질문 세트에서 반복 실행해 요약 지표를 만든다.
- **핵심 로직**: `run_evaluation_suite(repeats=2, persist_outputs=True)`가 baseline과 agent workflow를 여러 번 돌리고, 상세 결과 `results`와 집계표 `summary`를 반환한다.
- **주요 파라미터/변수**:
  - `repeats=2`: 같은 질문 세트를 두 번 반복해 평균 경향을 본다.
  - `persist_outputs=True`: 결과를 파일로 저장해 나중에 failure analysis에서 다시 쓸 수 있게 한다.
  - `summary`: 시스템별 평균 지표 표이다.

이 표에서는 한 지표만 보지 말고 여러 지표를 함께 읽어야 한다. 예를 들어 correctness가 조금 올라가도 grounding이 무너졌다면 실무에서는 오히려 더 위험할 수 있다.


In [ ]:
results, summary = run_evaluation_suite(repeats=2, persist_outputs=True)
summary

## baseline과 agent 비교

전체 평균만 보면 중요한 차이를 놓칠 수 있다. 이 셀은 시스템별, 질문 유형별로 지표를 다시 나누어 보여준다. 어떤 시스템은 simple lookup에는 강하지만 insufficient evidence 질문에서는 약할 수 있고, 반대로 abstain은 잘하지만 summary 품질이 떨어질 수도 있다.

- **목적**: 시스템 성능을 질문 유형별로 세분화해 읽는다.
- **핵심 로직**: `groupby(['system', 'expected_question_type'])`로 묶어 주요 지표 평균을 계산하고, 전체 `summary`와 함께 표시한다.
- **주요 파라미터/변수**:
  - `question_type_breakdown`: 시스템-질문유형 조합별 세부 성능표이다.
  - `expected_question_type`: 데이터셋이 기대하는 정답 유형이다.

이 결과를 볼 때는 예를 들어 `multi_hop`에서 retrieval_hit_rate는 괜찮은데 answer_correctness가 낮다면, 검색은 맞았지만 tool use나 synthesis가 부족했을 가능성을 생각해볼 수 있다. 이렇게 metric을 분해해서 읽으면 개선 우선순위가 명확해진다.


In [ ]:
question_type_breakdown = (
    results.groupby(['system', 'expected_question_type'])[[
        'answer_correctness',
        'retrieval_hit_rate',
        'grounding_pass_rate',
        'abstain_precision',
        'latency_seconds',
        'average_steps',
    ]]
    .mean()
    .round(3)
)

display(summary)
display(question_type_breakdown)

## 실험

레이더 차트(radar chart)는 여러 metric을 한눈에 비교하기 좋다. 다만 모양이 예뻐 보인다고 바로 결론을 내리면 안 된다. 각 축이 서로 다른 의미를 가지기 때문에, 어느 축이 왜 넓어졌는지를 함께 읽어야 한다. 여기서는 속도(latency)를 그대로 쓰지 않고 `speed_score`로 변환해 다른 지표와 같은 방향으로 해석할 수 있게 맞춘다.

- **목적**: baseline과 agent workflow의 다차원 성능 차이를 시각적으로 비교한다.
- **핵심 로직**: 요약표에서 주요 지표를 뽑아 레이더 차트로 그리고, `latency`는 역수 기반 `speed_score`로 변환한다.
- **주요 파라미터/변수**:
  - `radar_metrics`: 레이더 차트 축 목록이다.
  - `speed_score`: 지연 시간이 낮을수록 높은 점수를 받도록 만든 보조 지표이다.
  - `angles`: polar plot에서 각 축 위치를 잡기 위한 각도 배열이다.

레이더 차트를 읽을 때는 "면적이 큰 쪽이 무조건 승리"라고 해석하지 말고, grounding과 abstain precision처럼 안전성과 관련된 축이 얼마나 유지되는지를 먼저 보자. 실무에서는 속도보다 신뢰성이 우선되는 상황이 흔하다.


In [ ]:
radar = summary.set_index('system')[['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'abstain_precision']].copy()
radar['speed_score'] = 1.0 / summary.set_index('system')['latency'].clip(lower=0.001)
radar['speed_score'] = radar['speed_score'] / radar['speed_score'].max()
radar_metrics = list(radar.columns)
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
for system, row in radar.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, label=system)
    ax.fill(angles, values, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_title('Baseline vs Agent Workflow Radar View')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
radar.round(3)

## 결과 해석

이 셀은 agent workflow가 baseline 대비 얼마나 좋아졌거나 나빠졌는지를 차이값으로 계산한다. `agent_minus_baseline`이 양수면 agent가 그 지표에서 더 높은 값이고, 음수면 baseline이 더 높다는 뜻이다.

- **목적**: 두 시스템의 차이를 방향성과 크기까지 포함해 명확히 본다.
- **핵심 로직**: 시스템별 summary를 index로 맞춘 뒤, agent 행에서 baseline 행을 빼서 metric delta를 계산한다.
- **주요 파라미터/변수**:
  - `metric_deltas`: agent가 baseline보다 얼마나 달라졌는지를 담은 시리즈이다.
  - `agent_minus_baseline`: 해석을 돕기 위해 붙인 컬럼 이름이다.

이 표를 읽을 때는 보통 다음 순서가 좋다.
1. `grounding_pass_rate`, `abstain_precision`이 개선됐는지 본다. 안전성 관련 지표다.
2. `answer_correctness`와 `retrieval_hit_rate`를 본다. 실제 답 품질과 검색 품질이다.
3. `latency`, `average_steps`를 본다. 성능 향상을 위해 치른 비용이다.

어떤 metric을 올려야 시스템이 좋아지는가는 상황에 따라 다르다. 고객지원용 봇이라면 grounding과 abstain precision이 중요하고, 내부 탐색 도구라면 latency와 retrieval recall이 더 중요할 수 있다.


In [ ]:
metric_deltas = summary.set_index('system').loc['agent_workflow'] - summary.set_index('system').loc['baseline']
metric_deltas.to_frame(name='agent_minus_baseline').round(3)

## 핵심 정리

이 노트북을 통해 agent 시스템 평가는 단일 점수 경쟁이 아니라, 서로 다른 실패 모드를 분리해서 보는 작업이라는 점을 확인했다. correctness만 보면 좋아 보일 수 있지만, grounding이나 abstain precision까지 함께 보면 전혀 다른 그림이 나올 수 있다.

또한 질문 유형별 breakdown은 평균 점수보다 훨씬 실용적이다. 어느 유형에서 무너지는지 알아야 classifier, retriever, planner, verifier 중 어디를 먼저 고칠지 결정할 수 있다.

💡 면접 포인트: "evaluation은 모델 자랑이 아니라 의사결정 도구다. retrieval, grounding, abstention, latency를 함께 봐야 실제 운영 가능한 agent인지 판단할 수 있다"고 말하면 좋다.
